# LLM で文章を要約する（長さと観点を指定する）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/27_summarization/llm_summary.ipynb)

NLP クックブック 27-04 ／ 要約と平易化

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

文章の内容を、自分の言葉でまとめ直した要約（生成型要約）が欲しい。長さや、何に注目するかを指定したい。

In [1]:
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"   # 3B の日本語モデル。GPU（T4 など）で 1 要約あたり数秒〜10 秒ほど（VRAM 約 7GB）
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる（大規模言語モデル (LLM) の章の「オープンな LLM とチャットする（チャットテンプレート）」を参照）。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

def chat(prompt, max_new_tokens=150):
    inputs = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

text = ("市は 14 日、来年 4 月から市内の全小中学校で給食費を無償化すると発表した。対象は約 1 万 2 千人で、年間の費用は約 6 億円を見込む。"
        "財源には、ふるさと納税の寄付金と、公共施設の統廃合で生じる削減分を充てる。"
        "市長は記者会見で「子育て世帯の負担を減らし、若い世代に選ばれるまちにしたい」と述べた。"
        "一方、市議会の一部からは「恒久的な財源の裏付けが弱い」との指摘が出ている。"
        "市は来月の議会に関連する条例案を提出する方針で、可決されれば県内の市では 3 例目となる。"
        "保護者からは歓迎の声が上がる一方、食材費の高騰で給食の質が下がらないか心配する意見もある。")
print(len(text), "文字")
print(chat(f"次の文章を要約してください。\n\n{text}"))

279 文字
市は2024年4月から市内の全小中学校で給食費を無償化することを14日に発表しました。対象となるのはおよそ1万2千人で、年間費用は約6億円と見積もられています。この財源は、ふるさと納税の寄付金と公共施設の統廃合による削減金で賄われます。市長は記者会見で、この措置により子育て世帯の負担を軽減し、若い世代に選ばれるまちにしたいと述べました。しかし、市議会の一部からは恒久的な財源の裏付けが弱いとの指摘もあります。市は来月の議会に関連条例案を提出予定で、可決されれば県内では3例目の給食費無償化となります。保護者からは歓迎の声が上がる一方で、食材費高騰による給食の質の低下を懸念する意見もあります。


In [2]:
# 長さを指定する
for instruction in ["1 文 (40 字以内) で要約してください。", "100 字程度で要約してください。"]:
    summary = chat(f"次の文章を{instruction}要約だけを出力してください。\n\n{text}")
    print(f"[{instruction}] ({len(summary)} 字) {summary}")

[1 文 (40 字以内) で要約してください。] (87 字) 市は給食費を無償化し、約6億円の費用をふるさと納税寄付金と公共施設削減分で賄う。市長は負担軽減と若い世代誘致を目指すが、市議会一部からは財源の安定性に疑問の声が上がっている。
[100 字程度で要約してください。] (195 字) 市は市内の全小中学校で給食費を無償化することを決定した。これは約1万2千人が対象で、年間約6億円の費用がかかる。この財源はふるさと納税の寄付金と公共施設削減による削減分で賄われる。市長は子育て世帯の負担軽減と若い世代の誘致を目指すと述べている。しかし、市議会の一部からは恒久的な財源の懸念が指摘されている。保護者からは歓迎の声が上がる一方で、食材費高騰による給食の質低下を懸念する声もある。


In [3]:
# 観点や読み手を指定する
for focus in ["財源と費用に注目して、2 文で要約してください。", "賛成と反対の意見を 1 文ずつにまとめてください。", "小学生にもわかる言葉で、2 文で要約してください。"]:
    print(f"[{focus}]\n   {chat(focus + chr(10) + chr(10) + text)}")

[財源と費用に注目して、2 文で要約してください。]
   市は市内の全小中学校での給食費無償化を来年 4 月より実施予定で、年間約 6 億円の費用がかかると見込まれる。この財源はふるさと納税寄付金と公共施設削減による削減分で賄われ、市長は子育て世帯の負担軽減と若い世代の誘致を目指すが、市議会一部からは恒久的な財源の安定性について懸念の声が上がっている。
[賛成と反対の意見を 1 文ずつにまとめてください。]
   賛成の意見: 市は市内の全小中学校で給食費を無償化することで、子育て世帯の負担を軽減し、若い世代に選ばれるまちを実現しようとしている。

反対の意見: 給食費無償化の財源としてふるさと納税の寄付金と公共施設の削減分を当てているが、恒久的な財源の裏付けが弱く、持続可能性に疑問がある。
[小学生にもわかる言葉で、2 文で要約してください。]
   市は、来年4月から市内の小中学校で出す給食費を無料にすると決めました。対象となるのはおよそ1万2千人で、年間約6億円の費用がかかります。この費用は、ふるさと納税の寄付金と公共施設の削減分でまかなわれます。市長は子育て世帯の負担を減らしたいと話しており、県内では3番目の取り組みとなります。しかし、一部の議員からは財源の安定性について指摘もあります。また、保護者からは歓迎の声が上がる一方で、食材費の高騰による給食の質への影響を懸念する声もあります。


- 生成型要約は、文をつなぎ直したり言い換えたりできるので、抽出型より短く自然にまとまる。そのかわり、原文にないことを書く（ハルシネーション）おそれがある。数値と固有名詞は必ず原文と照らし合わせる（上の最初の出力でも、原文の「来年 4 月」が具体的な年に書き換えられている）。
- 文字数の指定は目安にしかならない。厳密な上限が必要なら、生成後に文字数を確かめて、超えていたら「もっと短く」と再生成する。
- 「誰のための、何のための要約か」を書くと質が上がる。同じ文章でも、目的が違えば残すべき情報が違う。
- 小さいモデルは、長い入力の後半を見落としやすい。長い文書は、この章の「長い文書を分割して要約する」を参照。